In [1]:
!pip -q install numpy scikit-learn sentence-transformers
!python --version
!pip freeze > requirements.txt

Python 3.13.18


In [2]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or not all(rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        self.records = records
        self.backend = backend
        corpus = [rec["text"] for rec in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(corpus)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(MODEL_NAME, device="cpu")
            self.check_length(corpus)
            self.matrix = self.model.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        limit = self.model.max_seq_length
        for item in texts:
            if len(self.model.tokenizer(item, truncation=False)["input_ids"]) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _scores(self, query):
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        scores = self._scores(query)
        if scores is None:
            return []
        wanted = filters or {}
        pool = [i for i, rec in enumerate(self.records)
                if all(rec.get(key) == val for key, val in wanted.items())]
        pool.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in pool[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for rec, score in hits:
        print(rec["id"], f"score={score:.3f}")
        print(rec["text"])

Writing lab6_search.py


In [3]:
%%writefile guided_chatbot.py
import argparse

from lab6_search import Retriever

FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    {"id": "faq:linkvalid",
     "question": "How long does the recovery link stay valid?",
     "answer": "The recovery link stays valid for thirty minutes.",
     "source": "it_help.txt"},
    {"id": "faq:fileformat",
     "question": "Which file format should I use for my final file?",
     "answer": "Submit the final file as a PDF.",
     "source": "assessment.txt"},
]


def build_bot(backend="tfidf"):
    rows = [{**entry, "text": entry["question"] + " " + entry["answer"]} for entry in FAQ]
    return Retriever(rows, backend)


def reply(engine, query):
    cleaned = query.strip()
    if not cleaned:
        return "Please enter a question."
    for entry in FAQ:
        if cleaned.casefold() == entry["question"].casefold():
            return f"{entry['answer']} [source: {entry['source']}]"
    matches = engine.search(query)
    if not matches:
        return "No matching FAQ. Please contact the help desk."
    out = ["Possible matches; choose or rephrase your question:"]
    out += [f"{rec['question']} (similarity {score:.3f})" for rec, score in matches]
    return "\n".join(out)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"], default="tfidf")
    options = parser.parse_args()
    engine = build_bot(options.backend)
    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            typed = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if typed.strip().lower() == "exit":
            break
        print("Bot:", reply(engine, typed))


if __name__ == "__main__":
    main()

Writing guided_chatbot.py


In [4]:
%%writefile create_dataset.py
import json
import sys
from pathlib import Path

entries = [
    ("late",
     "This synthetic handbook section describes how late work is handled in "
     "the teaching programme. Late submission is normally penalized. The "
     "penalty is ten percent of the available marks for each calendar day "
     "after the deadline, and work that is more than five days late receives "
     "no marks. Dr. Saeed Anwar coordinates the assessment office and "
     "publishes the final penalty table every semester. Students who miss a "
     "deadline for personal reasons should contact the coordinator as early "
     "as possible. Illness extensions require medical evidence before the "
     "deadline. The evidence must be a signed note from a licensed doctor "
     "that states the dates of the illness. Extensions for non-medical "
     "reasons need written approval from the course coordinator and are "
     "granted only in rare cases. An older version of this handbook allowed "
     "students to send medical evidence up to seven days after the "
     "deadline, but that rule was withdrawn and must not be used. Group "
     "assignments follow the same rules, and one extension request covers "
     "the whole group when every member is affected. Requests sent after "
     "marking has started cannot be accepted. All decisions are recorded in "
     "the student portal so that the history of each request can be checked "
     "later.",
     "Illness extensions require medical evidence before the deadline."),
    ("password",
     "This synthetic IT help section explains account access for students "
     "and staff. Reset a forgotten password using the recovery link. The "
     "link is shown below the login box on the sign-in page and is sent to "
     "the registered university email address. It stays valid for thirty "
     "minutes, and a new request cancels any earlier link. Never share "
     "recovery codes. The help desk will never ask for a password or a "
     "recovery code by phone, chat or email. If the registered email "
     "address is no longer available, the user must visit the IT desk with "
     "a student card so that the identity can be checked in person. Staff "
     "accounts are reset by the IT desk only, because they give access to "
     "grades and personal records. After three wrong password attempts the "
     "account is locked for fifteen minutes to protect it from guessing. "
     "Passwords should contain at least twelve characters and should not be "
     "reused on other websites. Students are encouraged to enable the "
     "second step sign-in option in their profile settings, which adds a "
     "code from a phone application to every new login.",
     "Reset a forgotten password using the recovery link."),
    ("assignment",
     "This synthetic assessment section explains how coursework is "
     "delivered. Upload assignments through the learning portal. Submit the "
     "final file as a PDF. Files in other formats are not opened by the "
     "markers, and a document that cannot be opened is treated as not "
     "submitted. The portal accepts files up to twenty megabytes, so large "
     "figures should be compressed before the upload, and the student must "
     "keep a copy of the confirmation message that appears after every "
     "successful upload because the confirmation is the only proof of "
     "submission that the office accepts during a dispute. Group projects "
     "are uploaded by the team leader only, and every team member must be "
     "named on the first page of the document. Laboratory reports for "
     "practical sessions are different because they are handed to the "
     "instructor on paper at the start of the next session and are not "
     "accepted through the portal. Handwritten work must be scanned clearly "
     "before it is uploaded, and photographs taken at an angle may be "
     "rejected by the coordinator. Submissions are accepted until midnight "
     "on the deadline day according to the portal clock, and a "
     "resubmission before the deadline replaces the earlier file without "
     "any penalty.",
     "Upload assignments through the learning portal."),
    ("contractor",
     "This synthetic workplace policy describes remote work rules for "
     "different groups of people. Employees may work remotely for up to two "
     "days each week when the line manager approves the plan in advance. "
     "Contractors can work remotely only if the signed contract permits it. "
     "Employee rules do not apply to contractors. A contractor whose "
     "contract is silent on remote work must work on site until the "
     "contract is amended by the human resources office. Visitors and "
     "interns follow a separate internship policy and should not use this "
     "section. Remote work from another country is not allowed for any "
     "group without written approval from the legal office, because tax and "
     "data protection rules differ between countries. Everyone who works "
     "remotely must use the company virtual private network and must keep "
     "work devices locked when they are not in use. Managers review remote "
     "work plans every quarter, and a plan can be withdrawn with one week "
     "of notice when the needs of the team change. Equipment such as "
     "laptops remains the property of the company and must be returned at "
     "the end of the arrangement.",
     "Contractors can work remotely only if the signed contract permits it."),
    ("printer",
     "This synthetic equipment guide collects short troubleshooting notes "
     "for office devices. For blank printer pages, check ink levels and run "
     "the cleaning utility. Print a test page afterwards. If the test page "
     "is still blank, open the printer menu and make sure that the correct "
     "paper size is selected, because a wrong size can cause empty output. "
     "Warning: switch the device off and wait ten minutes before touching "
     "any internal part, since the heating unit stays hot. For a blank "
     "scanner image, check document placement and select the correct "
     "scanning source. A scanner glass with fingerprints can also lower "
     "quality, so clean it with a soft dry cloth. A printer that makes "
     "noise but pulls no paper usually has a dirty roller, and the roller "
     "can be wiped with a slightly damp cloth after unplugging the cable. "
     "Streaks across the page point to a clogged print head, and a second "
     "cleaning cycle is usually enough. Each device model has its own "
     "manual, and the instructions for one model must not be applied to "
     "another model without checking. Replace cartridges only with the type "
     "listed in the manual.",
     "For blank printer pages, check ink levels and run the cleaning utility."),
]

corpus = {name: body for name, body, _ in entries}
gold = {name: ev for name, _, ev in entries}

qa_pairs = [
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("What are tomorrow's flight prices?", None, "test"),
    ("Who won the football match?", None, "test"),
]

labelled = [
    {"query": q, "doc": d, "evidence": gold[d] if d else None, "split": s}
    for q, d, s in qa_pairs
]

overwrite = "--force" in sys.argv
for filename, payload in (("corpus.json", corpus), ("queries.json", labelled)):
    target = Path(filename)
    if target.exists() and not overwrite:
        raise FileExistsError("Refusing to overwrite " + filename + " (use --force)")
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

for name, body in corpus.items():
    assert 150 <= len(body.split()) <= 250, name
    assert gold[name] in body
print("Created corpus.json and queries.json")
print({name: len(body.split()) for name, body in corpus.items()}, "words per document")

Writing create_dataset.py


In [5]:
!python create_dataset.py --force

Created corpus.json and queries.json
{'late': 198, 'password': 182, 'assignment': 196, 'contractor': 183, 'printer': 187} words per document


In [6]:
%%writefile task2_chunks.py
import json
import re
from pathlib import Path


def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    step = size - overlap
    out = []
    for first in range(0, len(spans), step):
        last = min(first + size, len(spans))
        out.append((offset + spans[first][0], offset + spans[last - 1][1]))
        if last == len(spans):
            break
    return out


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    bounds = [0, *(m.end() for m in re.finditer(r"[.!?]\s+", text)), len(text)]
    out = []
    current = None
    used = 0
    for lo, hi in zip(bounds, bounds[1:]):
        count = len(text[lo:hi].split())
        if count == 0:
            continue
        if count > size:
            if current:
                out.append(current)
            out.extend(windows(text[lo:hi], size, 0, lo))
            current, used = None, 0
        elif current and used + count <= size:
            current = (current[0], hi)
            used += count
        else:
            if current:
                out.append(current)
            current, used = (lo, hi), count
    if current:
        out.append(current)
    return out


def make_chunks(docs, strategy):
    rows = []
    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))
        for n, (lo, hi) in enumerate(spans):
            rows.append({"id": f"{doc_id}:{n}", "doc": doc_id,
                         "start": lo, "end": hi, "text": text[lo:hi]})
    return rows


RULE_PAIRS = {
    "late": ("Late submission is normally penalized.",
             "Illness extensions require medical evidence before the deadline."),
    "password": ("Reset a forgotten password using the recovery link.",
                 "Never share recovery codes."),
    "assignment": ("Upload assignments through the learning portal.",
                   "Submit the final file as a PDF."),
    "contractor": ("Contractors can work remotely only if the signed contract permits it.",
                   "Employee rules do not apply to contractors."),
    "printer": ("For blank printer pages, check ink levels and run the cleaning utility.",
                "Print a test page afterwards."),
}


def compare_strategies(docs):
    base_total = sum(len(t.split()) for t in docs.values())
    print(f"{'strategy':10} {'chunks':>6} {'avg words':>9} {'max words':>9} "
          f"{'dup words':>9} {'rule+exception together':>24}")
    for strategy in ("0", "10", "sentence"):
        rows = make_chunks(docs, strategy)
        lengths = [len(r["text"].split()) for r in rows]
        kept, marks = 0, []
        for doc, (rule, exc) in RULE_PAIRS.items():
            if doc not in docs:
                continue
            together = any(rule in r["text"] and exc in r["text"]
                           for r in rows if r["doc"] == doc)
            kept += together
            marks.append(f"{doc}={'Y' if together else 'N'}")
        print(f"{strategy:10} {len(rows):>6} {sum(lengths) / len(lengths):>9.1f} "
              f"{max(lengths):>9} {sum(lengths) - base_total:>9} "
              f"{kept}/{len(marks)}  ({', '.join(marks)})")


def run_checks():
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    lengthy = " ".join(["word"] * 95) + "."
    pieces = sentence_chunks(lengthy)
    assert all(len(lengthy[a:b].split()) <= 40 for a, b in pieces)
    assert len(pieces) == 3
    assert sentence_chunks("") == []
    sample = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ("0", "10", "sentence"):
        for chunk in make_chunks(sample, strategy):
            assert chunk["text"] == sample[chunk["doc"]][chunk["start"]:chunk["end"]]
            print(strategy, chunk)
    print("All boundary tests passed.\n")

    abbreviation = "Dr. Saeed Anwar coordinates the office. Contact him first."
    print("Abbreviation demo:")
    for a, b in sentence_chunks(abbreviation, size=5):
        print("  ", repr(abbreviation[a:b]))

    path = Path("corpus.json")
    if path.exists():
        corpus = json.loads(path.read_text(encoding="utf-8"))
        print("\nCorpus words per document:",
              {name: len(body.split()) for name, body in corpus.items()})
        print()
        compare_strategies(corpus)


if __name__ == "__main__":
    run_checks()

Writing task2_chunks.py


In [7]:
!python task2_chunks.py

0 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
10 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
sentence {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
All boundary tests passed.

Abbreviation demo:
   'Dr. '
   'Saeed Anwar coordinates the office. '
   'Contact him first.'

Corpus words per document: {'late': 198, 'password': 182, 'assignment': 196, 'contractor': 183, 'printer': 187}

strategy   chunks avg words max words dup words  rule+exception together
0              25      37.8        40         0 2/5  (late=N, password=N, assignment=Y, contractor=N, printer=Y)
10             32      38.0        40       270 3/5  (late=N, password=N, assignment=Y, contractor=Y, printer=Y)
sentence       36      26.3        40         0 3/5  (late=N, password=N, assignment

In [8]:
%%writefile guided_chunks.py
from task2_chunks import windows, sentence_chunks

passage = ("Late work is penalized. Illness may qualify for an extension "
           "when medical evidence is provided. Ask the coordinator first.")


def dump(spans):
    for begin, finish in spans:
        print(begin, finish, repr(passage[begin:finish]))
        assert 0 <= begin < finish <= len(passage)


print("word windows")
dump(windows(passage, size=12, overlap=4))
print("sentence aware")
dump(sentence_chunks(passage, size=12))

print("\n--- 20-word budget ---")
dump(sentence_chunks(passage, size=20))

print("\n--- sentence longer than the budget (size=12) ---")
lengthy = " ".join(["word"] * 30) + "."
for begin, finish in sentence_chunks(lengthy, size=12):
    print(begin, finish, len(lengthy[begin:finish].split()), "words")

Writing guided_chunks.py


In [10]:
!python guided_chunks.py

word windows
0 73 'Late work is penalized. Illness may qualify for an extension when medical'
48 122 'an extension when medical evidence is provided. Ask the coordinator first.'
sentence aware
0 24 'Late work is penalized. '
24 96 'Illness may qualify for an extension when medical evidence is provided. '
96 122 'Ask the coordinator first.'

--- 20-word budget ---
0 122 'Late work is penalized. Illness may qualify for an extension when medical evidence is provided. Ask the coordinator first.'

--- sentence longer than the budget (size=12) ---
0 59 12 words
60 119 12 words
120 150 6 words


In [9]:
%%writefile guided_embeddings.py
import numpy as np
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
passages = [
    "Reset a forgotten password using the recovery link.",
    "Submit assignments through the learning portal.",
    "Check ink levels when a printer produces blank pages.",
]
vectors = encoder.encode(passages, normalize_embeddings=True)


def rank(query):
    q = encoder.encode([query], normalize_embeddings=True)
    sims = (q @ vectors.T)[0]
    for idx in np.argsort(-sims):
        print(idx, round(float(sims[idx]), 3), passages[idx])


print("Embedding shape:", vectors.shape)
rank("How can I recover my login?")
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)

print("\nNegated query:")
rank("I do not need a password reset")

Writing guided_embeddings.py


In [11]:
!python guided_embeddings.py

modules.json: 100% 349/349 [00:00<00:00, 818kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 458kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 23.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 223kB/s]
config.json: 100% 612/612 [00:00<00:00, 1.96MB/s]

model.safetensors: downloading bytes:  92% 83.3M/90.9M [00:01<00:00, 96.2MB/s, 6.84MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 55.4MB/s, 8.02MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 59.2MB/s, 8.80MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 18973.75it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.52MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 21.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 56.0MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 501kB/s]
config.json: 100% 190/190 [00:00<00:00, 811kB/s]
Embedding shape: (3, 384)
0 0.627 Reset a forgotten password using the recovery link.
1 0.072 Submit as

In [12]:
%%writefile guided_limits.py
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")


def token_count(text):
    return len(encoder.tokenizer(text, truncation=False)["input_ids"])


for name, content in [("short", "Reset my password."), ("long", "password " * 300)]:
    total = token_count(content)
    print(name, "tokens:", total, "limit:", encoder.max_seq_length,
          "fits:", total <= encoder.max_seq_length)

print("\n--- whitespace vs model tokens ---")
for content in ["CS4944 late!", "help@example.com", "POST /v2/token/refresh"]:
    print(repr(content), "whitespace:", len(content.split()),
          "model tokens (with special):", token_count(content))

Writing guided_limits.py


In [13]:
!python guided_limits.py

Loading weights: 100% 103/103 [00:00<00:00, 6902.94it/s]
short tokens: 6 limit: 256 fits: True
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (302 > 256). Running this sequence through the model will result in indexing errors
long tokens: 302 limit: 256 fits: False

--- whitespace vs model tokens ---
'CS4944 late!' whitespace: 2 model tokens (with special): 7
'help@example.com' whitespace: 1 model tokens (with special): 7
'POST /v2/token/refresh' whitespace: 2 model tokens (with special): 11


In [14]:
%%writefile guided_suggestions.py
import argparse

from guided_chatbot import build_bot

parser = argparse.ArgumentParser()
parser.add_argument("--backend", choices=["tfidf", "embedding"], default="tfidf")
parser.add_argument("--query", default="assignment upload")
opts = parser.parse_args()

bot = build_bot(opts.backend)
print("Backend:", opts.backend, "| Query:", opts.query)
print("Suggested questions:")
for rec, score in bot.search(opts.query, k=3):
    if opts.backend == "embedding" or score > 0:
        print(rec["question"], f"({score:.3f})")

Writing guided_suggestions.py


In [15]:
!python guided_suggestions.py --backend tfidf --query "assignment upload"
!python guided_suggestions.py --backend embedding --query "forgot login"

Backend: tfidf | Query: assignment upload
Suggested questions:
Where do I submit my assignment? (0.594)
Loading weights: 100% 103/103 [00:00<00:00, 3362.91it/s]
Backend: embedding | Query: forgot login
Suggested questions:
How do I reset my password? (0.577)
How long does the recovery link stay valid? (0.170)
Can illness justify a late submission? (0.103)


In [16]:
%%writefile task2_semantic_check.py
import json
from pathlib import Path

from task2_chunks import make_chunks
from lab6_search import Retriever, show_hits

mini = {
    "it": "A forgotten password can be reset using the recovery link.",
    "assessment": "Assignments are uploaded through the learning portal.",
}
mini_records = make_chunks(mini, "sentence")
for backend in ("tfidf", "embedding"):
    print("Backend:", backend)
    show_hits(Retriever(mini_records, backend).search(
        "How can I regain access to my account?", k=2))

corpus_file = Path("corpus.json")
if corpus_file.exists():
    corpus = json.loads(corpus_file.read_text(encoding="utf-8"))
    pieces = make_chunks(corpus, "sentence")
    questions = [
        "How can I regain access to my account?",
        "What do I need if I fell sick and missed the deadline?",
        "Where should I hand in my coursework?",
    ]
    for backend in ("tfidf", "embedding"):
        engine = Retriever(pieces, backend)
        print("\n=== Backend:", backend, "===")
        for question in questions:
            print("\nQuery:", question)
            found = engine.search(question, k=3)
            if not found:
                print("  No matching records.")
            for rec, score in found:
                print(f"  {rec['id']} score={score:.3f} chars {rec['start']}-{rec['end']}")
                print("   ", rec["text"])

Writing task2_semantic_check.py


In [17]:
!python task2_semantic_check.py

Backend: tfidf
it:0 score=0.343
A forgotten password can be reset using the recovery link.
assessment:0 score=0.000
Assignments are uploaded through the learning portal.
Backend: embedding
Loading weights: 100% 103/103 [00:00<00:00, 3110.36it/s]
it:0 score=0.526
A forgotten password can be reset using the recovery link.
assessment:0 score=0.021
Assignments are uploaded through the learning portal.

=== Backend: tfidf ===

Query: How can I regain access to my account?
  password:0 score=0.262 chars 0-131
    This synthetic IT help section explains account access for students and staff. Reset a forgotten password using the recovery link. 
  password:4 score=0.258 chars 595-805
    Staff accounts are reset by the IT desk only, because they give access to grades and personal records. After three wrong password attempts the account is locked for fifteen minutes to protect it from guessing. 
  assignment:0 score=0.119 chars 0-152
    This synthetic assessment section explains how coursework 

In [18]:
import shutil
from google.colab import files
shutil.make_archive('Task2', 'zip', '.')
files.download('Task2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>